> # ⚠️ SUPERSEDED — do not run for reported numbers
>
> Use **`tiger_corrected_run.ipynb`** instead.
>
> This notebook runs against `master`, where the repair-measurement harness is
> broken in four independent ways (A1, A3/A3b, A5, A8 in `code_fixes/FIXES.md`).
> Numbers produced here cannot be reproduced or defended:
>
> * the "No Gamma Gate" ablation runs the identical configuration as Full System;
> * the random baseline is unseeded and not uniform;
> * "Restoration Accuracy" scores colour only — image repairs are not scored at all;
> * per-signal precision counts any dirty row as a hit.
>
> It also calls `--vlm-judge`, whose model ID is unverified (A2). On the fixed
> branch that now raises rather than silently vetoing every repair.
>
> Two statements in the cells below are simply wrong: the Arbiter is **logistic
> regression**, not "gradient boosting (XGBoost)"; and `data/sample/` could not
> be built at all until D5 was fixed.
>
> Kept for provenance — this is the notebook that produced the current tables.

# 🐯 TIGeR Kaggle Workflow
Welcome to the complete TIGeR (Text-Image Generative Repair) pipeline.

This notebook is broken down step-by-step so you can see exactly how the AI identifies, routes, and repairs multimodal errors in ecommerce catalogues.


## 📖 Glossary of Terms & Jargon

Before we begin, here is a quick guide to the terminology used in this pipeline:

* **TIGeR**: Text-Image Generative Repair (the name of this system).
* **V2T (Vision-to-Text)**: A repair strategy where we *fix the text* to match the image (e.g., updating the description to say "red shirt" instead of "blue shirt" because the image is clearly red).
* **T2V (Text-to-Vision)**: A repair strategy where we *replace the image* because the text description is correct, but the wrong photo was uploaded.
* **Sieve (Error Detection)**: The initial scanner that looks at the whole catalogue and flags products where the image and text do not seem to match.
* **Arbiter (Routing)**: The AI decision-maker. It looks at the flagged products and decides the safest way to fix them (V2T vs. T2V vs. Escalate to Human).
* **Solver (Repair Execution)**: The system that carries out the Arbiter's instructions (applying text patches or swapping images).
* **VLM Judge (Vision-Language Model)**: An advanced AI (like Google Gemini) that acts as a final safety checkpoint. It looks at the repaired product and vetoes the repair if it doesn't make sense.
* **Generative Fallback**: If the Arbiter wants to replace an image (T2V) but cannot find a valid replacement in the catalogue, the system uses Stable Diffusion to synthetically generate a brand new, mathematically perfect image from scratch.
* **LOO (Leave-One-Out) Masking**: A detective technique used by the system to figure out what is wrong. It hides one word at a time from the text to see if the image-text match improves, allowing it to pinpoint the exact wrong word.



### Step 0: Clone the Repository
First, we download the latest code from GitHub and set up our working directory.


In [ ]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!pip install -e ".[dev,vlm,gen]" -q

### Step 1: Environment Setup
We load the Gemini API key from Kaggle Secrets so our VLM Judge can verify complex repairs.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    gemini_key = user_secrets.get_secret("gemini api")
    os.environ["GEMINI_API_KEY"] = gemini_key
    print("✅ Gemini API Key loaded successfully from Kaggle Secrets!")
except Exception as e:
    print("❌ Failed to load Gemini API Key. Did you add it to Kaggle Secrets? Error:", e)

## Phase 1: Data Generation
### Step 2: Generate Synthetic Catalogue (`synthgen`)
Here we create a perfectly clean, ground-truth catalogue of synthetic products (shirts, shoes, bags, hats) and render their images. We also explicitly inject a highly unique item (a magenta velvet spacesuit) to guarantee a generative fallback scenario later.


In [ ]:
!python -m tiger.cli import-fashion --source /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset

## Phase 2: AI Calibration
### Step 3: Calibrate Sieve Thresholds (`calibrate`)
The system analyzes the clean data to learn what "normal" similarity scores look like for different product categories, setting automatic thresholds for error detection.


In [ ]:
!python -m tiger.cli calibrate

### Step 4: Train the AI Arbiter (`train-arbiter`)
We train a lightweight gradient boosting model (XGBoost) to act as the AI Arbiter. This model learns to look at error evidence and decide the safest repair strategy (e.g., Fix Text, Replace Image, or Escalate to Human).


In [ ]:
!python -m tiger.cli train-arbiter

### Step 5: Calibrate Decision Fusion (`calibrate-fusion`)
We tune the confidence gating of the Arbiter to ensure it hits a strict 85% precision floor, meaning it only automatically repairs items when it is highly confident.


In [ ]:
!python -m tiger.cli calibrate-fusion

## Phase 3: The Repair Pipeline
### Step 6: Inject Anomalies (`noise`)
Now we intentionally corrupt the clean catalogue! We drop images, swap images with other products, and mutate text attributes to simulate real-world e-commerce errors.


In [ ]:
!python -m tiger.cli noise --seed 7

### Step 7: Error Detection (`detect`)
The Sieve scans the corrupted catalogue and flags suspicious products that fail our calibrated similarity checks.


In [ ]:
!python -m tiger.cli detect --seed 7

### Step 8: Evidence Gathering (`analyze`)
For every flagged item, we gather deep evidence (leave-one-out masking, attribute probing) to figure out exactly *what* went wrong.


In [ ]:
!python -m tiger.cli analyze --seed 7

### Step 9: Routing Decisions (`route`)
The AI Arbiter reviews the evidence and decides how to fix each item (Fix Text vs. Replace Image).


In [ ]:
!python -m tiger.cli route --seed 7

### Step 10: Closed-Loop Repair (`repair`)
The Solver executes the Arbiter's plan. It attempts the fixes, asks the Gemini VLM Judge to verify them, and synthesizes brand new images via Stable Diffusion if no image exists in the catalogue.


In [ ]:
!python -m tiger.cli repair --seed 7 --vlm-judge --generative-fallback


## Phase 4: Outputs & Visualization
### Step 11: Export Data
Zip the outputs so you can download the repaired data and logs.


In [ ]:
!zip -r /kaggle/working/tiger_outputs.zip data/outputs data/thresholds data/processed
print("✅ Download tiger_outputs.zip from the '/kaggle/working' directory in the right sidebar!")

### Step 12: Visual Inspection Grid
Let's see the results! This cell plots a grid showing the original clean state, the corrupted state, and the final AI-repaired state. **Note: The forced Generative Fallback (the magenta spacesuit) is guaranteed to appear here!**


In [ ]:
# Visual Inspection: See the Clean -> Corrupted -> Repaired progression
from tiger.viz import plot_repair_stages
plot_repair_stages(seed=7)


### Step 15: Repair Ablation Study
This command takes 20 corrupted products and runs them through the repair cycle 5 different times, turning off a different component each time to see what breaks:

Full System: Runs normally as a baseline.
No Arbiter: Turns off our trained AI router and forces the system to guess randomly. (Proves the Arbiter is necessary).
No VLM Judge: Turns off Gemini. (Proves that without a VLM, the system accepts visually wrong images).
No Generative Fallback: Turns off Stable Diffusion. (Proves that without it, missing images become dead-ends).
No Gamma Gate: Forces the system to blindly trust the Arbiter even when it is uncertain. (Proves that our confidence threshold prevents bad automated repairs).

In [ ]:
!python -m tiger.cli ablate-repair --vlm-judge --generative-fallback 


In [ ]:
!python -m tiger.cli ablate-repair --independent --generative-fallback
